# LoRA Fine-Tuning

![](https://miro.medium.com/v2/resize:fit:700/1*bwbhjqxxC6IPKGxnmpVlwg.png)

This example template demonstrates **parameter-efficient fine-tuning (PEFT)** using **LoRA (Low-Rank Adaptation)** with the FLAN-T5 model on a free public dataset (SAMSum) for summarization.

This provides a lightweight, GPU-friendly workflow that runs fully offline — no API keys required. The notebook guides you through each step: loading data, applying LoRA adapters, fine-tuning, evaluating, and saving your model for reuse.

On [Saturn Cloud](https://saturncloud.io), you can scale from a single NVIDIA GPU to multi-GPU clusters, enabling distributed inference for larger models or higher throughput workloads — all within a managed, GPU-ready environment.

## Install dependencies

In [10]:
# Step 1: Install UV (fast, modern package manager)
!pip install -q uv
# Step 2: Create a clean environment with Python 3.12
!uv venv lora-env -p 3.12

# Step 3: Activate and install all required libraries inside it
!source lora-env/bin/activate && uv pip install -q torch transformers datasets peft accelerate evaluate bitsandbytes jedi

# Step 4: Add the environment as a selectable Jupyter kernel
!source lora-env/bin/activate && pip install -q ipykernel
!python -m ipykernel install --user --name=lora-env --display-name "LoRA Fine-Tune Env"

# (Optional fallback for environments without bitsandbytes)
try:
    import bitsandbytes
except Exception:
    print("⚠️ bitsandbytes not available — skipping GPU quantisation support.")


!pip install -q --upgrade \
    sentencepiece \
    protobuf \
    tqdm

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: lora-env
? A virtual environment already exists at `lora-env`. Do you want to replace it? [y/n] › yes

✔ A virtual environment already exists at `lora-env`. Do you want to replace it? · yes
Activate with: source lora-env/bin/activate
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Installed kernelspec lora-env in /root/.local/share/jupyter/kernels/lora-env


Download and prepares the GovReport Summarization dataset from `Hugging Face (ccdv/govreport-summarization)`. The dataset contains long government reports paired with their human-written summaries, making it suitable for text summarization tasks.

In [11]:
from datasets import load_dataset, Dataset
import pandas as pd

print("⏳ Downloading dataset: ccdv/govreport-summarization")
ds = load_dataset("ccdv/govreport-summarization")
train_ds = ds["train"].select(range(1000))
eval_ds  = ds["validation"].select(range(200))
TEXT_COL, TARGET_COL = "report", "summary"
print("✅ Dataset ready (govreport-summarization)")

⏳ Downloading dataset: ccdv/govreport-summarization
✅ Dataset ready (govreport-summarization)


Loads the **FLAN-T5-Small model** and its tokenizer from Hugging Face. The tokenizer converts text into numerical tokens the model can understand, while the model itself (a sequence-to-sequence language model) performs tasks such as summarization or text generation.

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
print(f"⏳ Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ Model and tokenizer loaded successfully!")
print("Tokenizer vocab size:", len(tokenizer))


⏳ Loading model: google/flan-t5-small
✅ Model and tokenizer loaded successfully!
Tokenizer vocab size: 32100


Adding LoRA (Low-Rank Adaptation) adapter to the base model using PEFT (Parameter-Efficient Fine-Tuning). Instead of updating all model parameters, LoRA inserts lightweight adapter layers that learn task-specific updates—making fine-tuning faster and more memory-efficient.

In [13]:
from peft import LoraConfig, get_peft_model

# LoRA configuration
lora_config = LoraConfig(
    r=16,                  # rank
    lora_alpha=32,         # scaling factor
    lora_dropout=0.05,     # dropout for regularisation
    bias="none",
    task_type="SEQ_2_SEQ_LM"  # T5-style sequence-to-sequence
)

# Apply adapter to model
model = get_peft_model(model, lora_config)

# Print summary
print("✅ LoRA adapter added successfully!")
model.print_trainable_parameters()


✅ LoRA adapter added successfully!
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


Prepare the text data for training by converting it into numerical tokens that the model can process.

In [14]:
def preprocess(examples):
    inputs = examples["report"]
    targets = examples["summary"]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing to the training and evaluation datasets
train_tok = train_ds.map(preprocess, batched=True)
eval_tok  = eval_ds.map(preprocess, batched=True)

print("✅ Tokenisation complete!")
print("Sample tokenised entry:")
print(train_tok[0])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Tokenisation complete!
Sample tokenised entry:
{'report': 'The structure of the armed forces is based on the Total Force concept, which recognizes that all elements of the structure—active duty military personnel, reservists, defense contractors, host nation military and civilian personnel, and DOD federal civilian employees—contribute to national defense. In recent years, federal civilian personnel have deployed along with military personnel to participate in Operations Joint Endeavor, conducted in the countries of Bosnia-Herzegovina, Croatia, and Hungary; Joint Guardian, in Kosovo; and Desert Storm, in Southwest Asia. Further, since the beginning of the Global War on Terrorism, the role of DOD’s federal civilian personnel has expanded to include participation in combat support functions in Operations Enduring Freedom and Iraqi Freedom. DOD relies on the federal civilian personnel it deploys to support a range of essential missions, including intelligence collection, criminal invest

Setting up the training loop that fine-tunes the model using Hugging Face’s Seq2SeqTrainer API.

In [15]:
import torch
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Prepare data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Define training arguments
args = Seq2SeqTrainingArguments(
    output_dir="outputs-lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU supports it
    report_to=[],  # disables online tracking (no API needed)
)

# Initialise trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("🚀 Starting fine-tuning…")
trainer.train()
print("✅ Training complete!")

/tmp/ipython-input-2964402256.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


🚀 Starting fine-tuning…


Step,Training Loss
25,0.000000
50,0.000000
75,0.000000
100,0.000000
125,0.000000
150,0.000000
175,0.000000
200,0.000000
225,0.000000
250,0.000000


✅ Training complete!


Let's test the fine-tuned model to verify that it can generate meaningful summaries. It performs a full inference pass using the model and tokenizer.

In [16]:
test_input = "Write a brief summary: Alice and Bob discussed weekend plans. Bob suggested hiking, but Alice preferred visiting the museum."

# Tokenise and move to model device
inputs = tokenizer(test_input, return_tensors="pt", truncation=True, padding=True).to(model.device)

# Generate output
outputs = model.generate(**inputs, max_new_tokens=80)

# Decode and display
print("\n🧠 Fine-tuned Model Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))



🧠 Fine-tuned Model Output:

Bob and Alice discuss the museum's history.


This allows interactively test the fine-tuned model with your own custom input.

In [17]:
print("💬 Try your own prompt!")

user_prompt = input("\nEnter a text or paragraph you'd like the model to summarise: ")

# Tokenise user prompt
inputs = tokenizer(user_prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)

# Generate output
outputs = model.generate(**inputs, max_new_tokens=80)

# Decode and print
print("\n🧩 Model Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


💬 Try your own prompt!

Enter a text or paragraph you'd like the model to summarise: what is it doing 

🧩 Model Output:

It is doing it doing it doing it


In this template, you fine-tuned **Google’s FLAN-T5-Small** model using **LoRA (Low-Rank Adaptation)** with the **PEFT** library — a modern, lightweight approach to large language model adaptation.

Running this workflow on **Saturn Cloud** makes it both **scalable and cost-effective**. Saturn Cloud’s managed infrastructure allows you to:

* Start with a **single NVIDIA GPU** for experimentation and scale up to multi-GPU clusters for larger models.
* Collaborate across teams easily through shared Jupyter environments.
* Integrate this fine-tuning workflow into production pipelines for enterprise-ready deployment.

By using this template, you now have a complete, ready-to-run foundation for **adapter-based fine-tuning** in Saturn Cloud — ideal for tasks like summarisation, translation, or instruction-following with minimal resource use.

To continue exploring, check out:

* [Saturn Cloud Documentation](https://saturncloud.io/docs/) — for advanced configuration and GPU scaling.
* [Saturn Cloud Templates](https://saturncloud.io/resources/templates/) — for more examples of ML, LLM, and data science workflows.